# Step 4 — Does respecting the retention windows help? (windowed vs naive history features)

The interim feedback asked how the history timeline should inform feature engineering. This step tests the answer: the same ranker trained on the same users with history features built two ways.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [ ]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/clarequek/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

## Design (`scripts/02d_window_ablation.py`, `src/jds/window_ablation.py`)

* **windowed (pipeline)** — every count is per type inside that type's window + days since last event.
* **naive** — counts per type over the whole history, a 30-day half-life decayed score that mixes types
  (click 1, cart 2, order 3), and per-user click/cart/order ratios (the previous team's statistics).
* **windowed + naive ratios** — do the ratios add anything on top?
* **no history counts** — lower bound.

Everything else (query → item, CTR, lexical, slate context, item content) is identical and no tower score is used, so
this runs on CPU right after step 2. Rankers are trained on `diagnostics.ablation_user_frac` (30%) of training users
and evaluated on all test users. The right panel splits test users by how much of their history is older than the
7-day click window — where naive counts should mislead most.

In [ ]:
run.stage("02d_window_ablation.py")

In [ ]:
banner()
run.table("window_ablation.csv", caption="Same ranker, different history features (test users)")

In [ ]:
run.figure("window_ablation.png")
run.table("window_ablation_segments.csv", highlight=("mrr", "gndcg10"))